<a href="https://colab.research.google.com/github/jonasknoll57/Bachelorarbeit_Demand-AD/blob/main/V1_Source_Target.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Zugangsberechtigung auf Drive

from google.colab import drive
drive.mount('/content/drive')

# Gezippte Daten werden entpackt und in hohes Verzeichnis gelegt

!cp "/content/drive/MyDrive/BA_Colab/data.zip" "/content/data.zip"

!unzip -q "/content/data.zip" -d "/content"

!rm "/content/data.zip"
!rm "/content/_MACOSX"


Mounted at /content/drive
Mounted at /content/drive
rm: cannot remove '/content/_MACOSX': No such file or directory
rm: cannot remove '/content/_MACOSX': No such file or directory


Aus letzten Veruschen sind folgende Städte bekannt:

[Heidelberg, Marburg, Gießen, Freiburg, Leverkusen, Braunschweig, Kaiserslautern, Dortmund, Kassel, Duisburg, Ludwigshafen, Bochum, Wiesbaden, Winsen (Luhe), Essen, Offenburg]


In [7]:
import pandas as pd
import numpy as np
import glob, os, re
import warnings
warnings.filterwarnings('ignore')

# ── Pfade ──────────────────────────────────────────────────────
DEMAND_BASE    = '/content/data/demand'
STATION_NAMES_PATH = '/content/data/station_names/station_names.parquet'

# ── Kandidaten aus Schritt 1 (station_based) ───────────────────
CANDIDATES = [
    'Mannheim', 'Heidelberg', 'Marburg', 'Gießen', 'Freiburg',
    'Leverkusen', 'Braunschweig', 'Kaiserslautern', 'Dortmund',
    'Kassel', 'Duisburg', 'Ludwigshafen', 'Bochum', 'Wiesbaden',
    'Winsen (Luhe)', 'Essen', 'Offenburg',
]

# ── Schwellenwerte ─────────────────────────────────────────────
SOURCE_MIN_MONTHS   = 12
SOURCE_MIN_COVERAGE = 90.0
TARGET_MIN_MONTHS   = 3
TARGET_MIN_COVERAGE = 80.0
MAX_GAP_DAYS        = 30

# ── Mindest-EPD (Events Per Day) pro Station ───────────────────
# EPD = (n_lends + n_returns) einer Station, gemittelt über alle aktiven Tage.
# Bedeutung: Wie viele Ausleih- + Rückgabe-Ereignisse passieren im Schnitt
# pro Tag an einer Station?
# Warum wichtig für AD: Eine Station mit < 1 EPD liefert an den meisten Tagen
# den Wert 0 → kein auswertbares Signal, Anomalien nicht erkennbar.
MIN_EPD = 1.0

In [8]:
station_names = pd.read_parquet(STATION_NAMES_PATH)
station_names = station_names.rename(columns={'id': 'station_name_id', 'name': 'station_name'})

def classify_station(name: str) -> str:
    """Klassifiziert einen Stationsnamen in einen von 5 Typen."""
    if not isinstance(name, str) or name.strip() == '':
        return 'unknown'
    n = name.strip()
    if re.search(r'(?i)^recording[_\-\s]', n):  return 'recording'
    if re.match(r'(?i)^bike[-_]?\s*\d*', n):    return 'bike'
    if re.search(r'(?i)(virtuell|virtual)', n):  return 'virtual'
    if re.fullmatch(r'[\d\s\-_\.#/]+', n):      return 'only_nums'
    return 'real'

station_names['station_type'] = station_names['station_name'].apply(classify_station)

# Lookup-Dict: station_name_id → station_type  (für schnellen Join)
type_lookup = station_names.set_index('station_name_id')['station_type'].to_dict()

print('Station-Typ-Verteilung in station_names:')
print(station_names['station_type'].value_counts().to_string())

Station-Typ-Verteilung in station_names:
station_type
bike         44747
real         13040
virtual       4827
recording     1972
only_nums       51


In [9]:
def load_city(city: str, base: str) -> pd.DataFrame | None:
    """Lädt alle Parquet-Dateien aus dem Stadtordner rekursiv."""
    pattern = os.path.join(base, city, '**', '*.parquet')
    files   = glob.glob(pattern, recursive=True)
    if not files:
        pattern = os.path.join(base, city, '*.parquet')
        files   = glob.glob(pattern)
    if not files:
        return None

    cols  = ['network_name', 'timestamp', 'station_id', 'station_name_id', 'n_lends', 'n_returns']
    parts = [pd.read_parquet(f, columns=cols) for f in files]
    return pd.concat(parts, ignore_index=True)


city_dfs_raw: dict[str, pd.DataFrame] = {}
missing = []

for city in CANDIDATES:
    df = load_city(city, DEMAND_BASE)
    if df is None:
        missing.append(city)
        print(f'  ⚠️  {city}: nicht gefunden')
        continue

    df['timestamp']     = pd.to_datetime(df['timestamp'], utc=True)
    df['date']          = df['timestamp'].dt.date

    # Station-Typ aus Lookup anhängen (kein teurer Merge nötig)
    df['station_type']  = df['station_name_id'].map(type_lookup).fillna('unknown')

    city_dfs_raw[city]  = df
    n_real = (df['station_type'] == 'real').sum()
    n_bike = (df['station_type'] == 'bike').sum()
    n_rec  = (df['station_type'] == 'recording').sum()
    print(f'  ✅ {city}: {len(df):>10,} Zeilen  '
          f'(real={n_real:,}  bike={n_bike:,}  recording={n_rec:,})')

print(f'\n→ {len(city_dfs_raw)} Städte geladen, {len(missing)} fehlen: {missing}')

  ✅ Mannheim:  2,683,584 Zeilen  (real=2,579,329  bike=103,497  recording=754)
  ✅ Heidelberg:  1,957,371 Zeilen  (real=1,873,100  bike=84,206  recording=65)
  ✅ Marburg:  1,908,545 Zeilen  (real=1,837,213  bike=71,332  recording=0)
  ✅ Gießen:  1,082,120 Zeilen  (real=1,041,980  bike=40,091  recording=49)
  ✅ Freiburg:    904,037 Zeilen  (real=872,531  bike=31,335  recording=171)
  ✅ Leverkusen:    631,820 Zeilen  (real=602,444  bike=29,145  recording=229)
  ✅ Braunschweig:    420,448 Zeilen  (real=399,470  bike=20,924  recording=32)
  ✅ Kaiserslautern:    376,263 Zeilen  (real=357,303  bike=18,820  recording=138)
  ✅ Dortmund:    345,749 Zeilen  (real=334,022  bike=11,471  recording=256)
  ✅ Kassel:    337,986 Zeilen  (real=323,835  bike=14,044  recording=106)
  ✅ Duisburg:    253,832 Zeilen  (real=246,265  bike=7,419  recording=148)
  ✅ Ludwigshafen:    243,156 Zeilen  (real=227,788  bike=15,309  recording=57)
  ✅ Bochum:    203,035 Zeilen  (real=195,168  bike=7,750  recording=117)


In [10]:
KEEP_TYPES = {'real'}   # ← nur echte Stationen; 'virtual' bei Bedarf ergänzen

city_dfs: dict[str, pd.DataFrame] = {}

print(f'{'Stadt':<20} {'Gesamt':>10} {'→ real':>10} {'Anteil':>8}  Verworfen')
print('-' * 65)

for city, df in city_dfs_raw.items():
    df_real  = df[df['station_type'].isin(KEEP_TYPES)].copy()
    n_before = len(df)
    n_after  = len(df_real)
    pct      = n_after / n_before * 100 if n_before > 0 else 0

    # Aufschlüsselung der verworfenen Zeilen
    dropped = df[~df['station_type'].isin(KEEP_TYPES)]['station_type'].value_counts().to_dict()

    city_dfs[city] = df_real
    print(f'{city:<20} {n_before:>10,} {n_after:>10,} {pct:>7.1f}%  {dropped}')

Stadt                    Gesamt     → real   Anteil  Verworfen
-----------------------------------------------------------------
Mannheim              2,683,584  2,579,329    96.1%  {'bike': 103497, 'recording': 754, 'only_nums': 4}
Heidelberg            1,957,371  1,873,100    95.7%  {'bike': 84206, 'recording': 65}
Marburg               1,908,545  1,837,213    96.3%  {'bike': 71332}
Gießen                1,082,120  1,041,980    96.3%  {'bike': 40091, 'recording': 49}
Freiburg                904,037    872,531    96.5%  {'bike': 31335, 'recording': 171}
Leverkusen              631,820    602,444    95.4%  {'bike': 29145, 'recording': 229, 'only_nums': 2}
Braunschweig            420,448    399,470    95.0%  {'bike': 20924, 'recording': 32, 'virtual': 22}
Kaiserslautern          376,263    357,303    95.0%  {'bike': 18820, 'recording': 138, 'only_nums': 2}
Dortmund                345,749    334,022    96.6%  {'bike': 11471, 'recording': 256}
Kassel                  337,986    323,835   

In [11]:
def temporal_coverage(city: str, df: pd.DataFrame) -> dict:
    min_date    = df['timestamp'].min()
    max_date    = df['timestamp'].max()
    total_days  = max(1, (max_date - min_date).days)
    active_days = df['date'].nunique()
    coverage    = active_days / total_days * 100
    months      = total_days / 30.44

    # Größte Lücke
    sorted_dates = sorted(df['date'].unique())
    gaps = [
        (pd.Timestamp(sorted_dates[i]) - pd.Timestamp(sorted_dates[i-1])).days - 1
        for i in range(1, len(sorted_dates))
    ]
    max_gap = max(gaps, default=0)

    if months >= SOURCE_MIN_MONTHS and coverage >= SOURCE_MIN_COVERAGE and max_gap <= MAX_GAP_DAYS:
        role = 'SOURCE'
    elif months >= TARGET_MIN_MONTHS and coverage >= TARGET_MIN_COVERAGE:
        role = 'TARGET'
    elif months >= TARGET_MIN_MONTHS:
        role = 'TARGET_LOW_COV'
    else:
        role = 'EXCLUDE'

    return dict(city=city, min_date=min_date.date(), max_date=max_date.date(),
                total_days=total_days, total_months=round(months, 1),
                active_days=active_days, coverage_pct=round(coverage, 1),
                max_gap_days=max_gap, role=role)


coverage_rows = [temporal_coverage(c, df) for c, df in city_dfs.items()]
df_coverage   = pd.DataFrame(coverage_rows).sort_values('total_months', ascending=False)

icon_map = {'SOURCE': '✅ SOURCE', 'TARGET': '✅ TARGET',
            'TARGET_LOW_COV': '⚠️ TARGET (Coverage)', 'EXCLUDE': '❌ EXCLUDE'}

display(
    df_coverage.assign(role=df_coverage['role'].map(icon_map))
    .style
    .format({'coverage_pct': '{:.1f}%', 'total_months': '{:.1f}'})
    .bar(subset=['coverage_pct'], color='#90EE90', vmin=0, vmax=100)
    .bar(subset=['max_gap_days'], color='#FFB6C1', vmin=0)
)

,city,min_date,max_date,total_days,total_months,active_days,coverage_pct,max_gap_days,role
0,Mannheim,2023-03-31,2026-02-02,1039,34.1,1039,100.0%,1,✅ SOURCE
1,Heidelberg,2023-03-31,2026-02-02,1039,34.1,1039,100.0%,1,✅ SOURCE
2,Marburg,2023-03-31,2026-02-02,1039,34.1,1008,97.0%,20,✅ SOURCE
3,Gießen,2023-03-31,2026-02-02,1039,34.1,1008,97.0%,20,✅ SOURCE
5,Leverkusen,2023-03-31,2026-02-02,1039,34.1,1008,97.0%,20,✅ SOURCE
7,Kaiserslautern,2023-03-31,2026-02-02,1039,34.1,1039,100.0%,1,✅ SOURCE
14,Winsen (Luhe),2023-04-01,2026-02-02,1038,34.1,1007,97.0%,20,✅ SOURCE
11,Ludwigshafen,2023-03-31,2026-02-02,1039,34.1,1039,100.0%,1,✅ SOURCE
9,Kassel,2023-03-31,2026-02-02,1039,34.1,1008,97.0%,20,✅ SOURCE
13,Wiesbaden,2023-03-31,2026-02-02,1039,34.1,1008,97.0%,20,✅ SOURCE


In [12]:
def demand_volume(city: str, df: pd.DataFrame) -> dict:
    """
    Berechnet EPD-Kennzahlen ausschließlich auf Basis echter Stationen.
    df ist bereits auf station_type == 'real' gefiltert.
    """
    n_stations   = df['station_id'].nunique()
    n_days       = df['date'].nunique()
    total_events = df['n_lends'].sum() + df['n_returns'].sum()

    # Netzwerk-Durchschnitt (durch Hub-Stationen verzerrt!)
    avg_epd_network = total_events / n_stations / n_days if (n_stations * n_days) > 0 else 0

    # EPD je Station → robustere Verteilungskennzahlen
    per_station = (
        df.groupby('station_id')
        .apply(lambda x: (x['n_lends'].sum() + x['n_returns'].sum()) / n_days)
        .reset_index(name='epd')
    )

    # Anteil Stationen unterhalb Mindest-EPD
    # → hoher Anteil = viele Stationen zu inaktiv für AD
    low_signal_pct = (per_station['epd'] < MIN_EPD).mean() * 100

    median_epd = per_station['epd'].median()
    p25_epd    = per_station['epd'].quantile(0.25)   # 25% der Stationen haben EPD ≤ dieser Wert
    p75_epd    = per_station['epd'].quantile(0.75)

    # Signal gut, wenn: Median ≥ MIN_EPD UND weniger als 50% der Stationen zu schwach
    signal_ok  = median_epd >= MIN_EPD and low_signal_pct < 50

    return dict(
        city=city,
        n_real_stations=n_stations,
        avg_epd=round(avg_epd_network, 2),
        median_epd=round(median_epd, 2),
        p25_epd=round(p25_epd, 2),
        p75_epd=round(p75_epd, 2),
        low_signal_pct=round(low_signal_pct, 1),  # % Stationen mit EPD < MIN_EPD
        signal_ok=signal_ok,
    )


volume_rows = [demand_volume(c, df) for c, df in city_dfs.items()]
df_volume   = pd.DataFrame(volume_rows).sort_values('median_epd', ascending=False)

display(
    df_volume.assign(signal_ok=df_volume['signal_ok'].map({True: '✅ OK', False: '⚠️ SCHWACH'}))
    .style
    .format({'avg_epd': '{:.2f}', 'median_epd': '{:.2f}',
             'p25_epd': '{:.2f}', 'p75_epd': '{:.2f}', 'low_signal_pct': '{:.1f}%'})
    .bar(subset=['avg_epd', 'median_epd'], color='#ADD8E6')
    .bar(subset=['low_signal_pct'], color='#FFB6C1', vmin=0, vmax=100)
)

,city,n_real_stations,avg_epd,median_epd,p25_epd,p75_epd,low_signal_pct,signal_ok
2,Marburg,61,43.43,29.78,7.90,59.47,8.2%,✅ OK
1,Heidelberg,70,35.62,23.92,9.86,45.42,15.7%,✅ OK
4,Freiburg,132,31.05,21.74,6.80,38.85,9.1%,✅ OK
0,Mannheim,128,26.56,16.91,2.12,41.41,18.8%,✅ OK
3,Gießen,67,21.35,12.73,1.56,28.88,22.4%,✅ OK
8,Dortmund,100,14.36,9.96,3.74,17.65,7.0%,✅ OK
10,Duisburg,58,18.38,8.78,2.82,19.91,6.9%,✅ OK
14,Winsen (Luhe),22,8.59,7.01,2.81,10.04,9.1%,✅ OK
7,Kaiserslautern,37,11.27,6.76,0.80,17.94,29.7%,✅ OK
12,Bochum,95,8.67,4.97,2.38,12.39,9.5%,✅ OK


In [13]:
df_summary = df_coverage.merge(
    df_volume[['city', 'n_real_stations', 'avg_epd', 'median_epd', 'low_signal_pct', 'signal_ok']],
    on='city'
)

def recommend(row):
    if row['role'] == 'SOURCE' and row['signal_ok']:      return 'SOURCE'
    if row['role'] in ('TARGET', 'TARGET_LOW_COV') and row['signal_ok']: return 'TARGET'
    return 'EXCLUDE'

df_summary['recommendation'] = df_summary.apply(recommend, axis=1)

print('=' * 65)
for rec, label, icon in [
    ('SOURCE',  'Source-Städte  (≥12 Mo, Cov ≥90%, Signal OK)', '🟢'),
    ('TARGET',  'Target-Städte  (≥3 Mo,  Cov ≥80%, Signal OK)', '🔵'),
    ('EXCLUDE', 'Ausgeschlossen', '🔴'),
]:
    subset = df_summary[df_summary['recommendation'] == rec]
    print(f'\n{icon} {label} ({len(subset)}):')
    for _, r in subset.iterrows():
        warn = []
        if r['coverage_pct'] < SOURCE_MIN_COVERAGE: warn.append(f'Cov={r["coverage_pct"]:.0f}%')
        if r['max_gap_days'] > MAX_GAP_DAYS:         warn.append(f'Gap={r["max_gap_days"]}d')
        if not r['signal_ok']:                        warn.append(f'EPD↓')
        warn_str = '  ⚠️ ' + ', '.join(warn) if warn else ''
        print(f"   • {r['city']:<20s} | {r['total_months']:5.1f} Mo | "
              f"Cov {r['coverage_pct']:5.1f}% | "
              f"Median-EPD {r['median_epd']:.2f} | "
              f"{r['n_real_stations']} Stationen"
              f"{warn_str}")

print('\n' + '=' * 65)


🟢 Source-Städte  (≥12 Mo, Cov ≥90%, Signal OK) (11):
   • Mannheim             |  34.1 Mo | Cov 100.0% | Median-EPD 16.91 | 128 Stationen
   • Heidelberg           |  34.1 Mo | Cov 100.0% | Median-EPD 23.92 | 70 Stationen
   • Marburg              |  34.1 Mo | Cov  97.0% | Median-EPD 29.78 | 61 Stationen
   • Gießen               |  34.1 Mo | Cov  97.0% | Median-EPD 12.73 | 67 Stationen
   • Leverkusen           |  34.1 Mo | Cov  97.0% | Median-EPD 2.46 | 152 Stationen
   • Kaiserslautern       |  34.1 Mo | Cov 100.0% | Median-EPD 6.76 | 37 Stationen
   • Winsen (Luhe)        |  34.1 Mo | Cov  97.0% | Median-EPD 7.01 | 22 Stationen
   • Ludwigshafen         |  34.1 Mo | Cov 100.0% | Median-EPD 2.77 | 65 Stationen
   • Kassel               |  34.1 Mo | Cov  97.0% | Median-EPD 2.55 | 77 Stationen
   • Wiesbaden            |  34.1 Mo | Cov  97.0% | Median-EPD 4.84 | 36 Stationen
   • Offenburg            |  30.5 Mo | Cov  98.8% | Median-EPD 4.09 | 28 Stationen

🔵 Target-Städte  (≥3 Mo,  

In [14]:
df_summary.to_csv('network_analysis_results.csv', index=False)
print('Gespeichert: network_analysis_results.csv')
df_summary

Gespeichert: network_analysis_results.csv


,city,min_date,max_date,total_days,total_months,active_days,coverage_pct,max_gap_days,role,n_real_stations,avg_epd,median_epd,low_signal_pct,signal_ok,recommendation
0,Mannheim,2023-03-31,2026-02-02,1039,34.1,1039,100.0,1,SOURCE,128,26.56,16.91,18.8,True,SOURCE
1,Heidelberg,2023-03-31,2026-02-02,1039,34.1,1039,100.0,1,SOURCE,70,35.62,23.92,15.7,True,SOURCE
2,Marburg,2023-03-31,2026-02-02,1039,34.1,1008,97.0,20,SOURCE,61,43.43,29.78,8.2,True,SOURCE
3,Gießen,2023-03-31,2026-02-02,1039,34.1,1008,97.0,20,SOURCE,67,21.35,12.73,22.4,True,SOURCE
4,Leverkusen,2023-03-31,2026-02-02,1039,34.1,1008,97.0,20,SOURCE,152,4.79,2.46,32.2,True,SOURCE
5,Kaiserslautern,2023-03-31,2026-02-02,1039,34.1,1039,100.0,1,SOURCE,37,11.27,6.76,29.7,True,SOURCE
6,Winsen (Luhe),2023-04-01,2026-02-02,1038,34.1,1007,97.0,20,SOURCE,22,8.59,7.01,9.1,True,SOURCE
7,Ludwigshafen,2023-03-31,2026-02-02,1039,34.1,1039,100.0,1,SOURCE,65,4.02,2.77,24.6,True,SOURCE
8,Kassel,2023-03-31,2026-02-02,1039,34.1,1008,97.0,20,SOURCE,77,5.47,2.55,24.7,True,SOURCE
9,Wiesbaden,2023-03-31,2026-02-02,1039,34.1,1008,97.0,20,SOURCE,36,5.58,4.84,11.1,True,SOURCE
